<a href="https://colab.research.google.com/github/SahilRathi-AI/Gender-Bias-Salary-Ethics-Audit/blob/main/Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import glob
import gc

import numpy as np
import pandas as pd

In [5]:
data_path = "/content/drive/MyDrive/CICIDS_Data/*.csv"

all_csv_files = glob.glob(data_path, recursive=True)

cicids2017_files = []

for file_path in all_csv_files:
    file_name = os.path.basename(file_path)

    if "pcap_ISCX" in file_name:
        cicids2017_files.append(file_path)

print("CICIDS2017 files found:", len(cicids2017_files))

for file_path in cicids2017_files:
    print(os.path.basename(file_path))

NameError: name 'glob' is not defined

In [6]:
dataframes = []

for file_path in cicids2017_files:
    print("Loading:", os.path.basename(file_path))

    temp_data = pd.read_csv(file_path, low_memory=False)

    # remove extra spaces from column names
    temp_data.columns = temp_data.columns.str.strip()

    dataframes.append(temp_data)

cicids2017_data = pd.concat(dataframes, ignore_index=True)

del dataframes
gc.collect()

print("CICIDS2017 loaded successfully.")
print("Dataset shape:", cicids2017_data.shape)

NameError: name 'cicids2017_files' is not defined

In [7]:
print("Label column exists:", "Label" in cicids2017_data.columns)

print("\nLabel distribution:")
print(cicids2017_data["Label"].value_counts())

NameError: name 'cicids2017_data' is not defined

In [8]:
cicids2017_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values before cleaning:", cicids2017_data.isnull().sum().sum())

cicids2017_data.dropna(inplace=True)

print("Shape after cleaning:", cicids2017_data.shape)
print("Missing values after cleaning:", cicids2017_data.isnull().sum().sum())

NameError: name 'cicids2017_data' is not defined

In [9]:
cicids2017_data["Label_binary"] = cicids2017_data["Label"].apply(
    lambda label: 0 if str(label).upper() == "BENIGN" else 1
)

print("\nBinary label distribution:")
print(cicids2017_data["Label_binary"].value_counts())

NameError: name 'cicids2017_data' is not defined

In [10]:
benign_traffic_2017 = cicids2017_data[
    cicids2017_data["Label_binary"] == 0
].copy()

print("Benign traffic shape:", benign_traffic_2017.shape)

NameError: name 'cicids2017_data' is not defined

In [11]:
X_train_2017 = benign_traffic_2017.drop(
    columns=["Label", "Label_binary"],
    errors="ignore"
)

X_train_2017 = X_train_2017.select_dtypes(include=["number"])

training_features = X_train_2017.columns.tolist()

print("Training data shape:", X_train_2017.shape)
print("Number of training features:", len(training_features))

NameError: name 'benign_traffic_2017' is not defined

In [12]:
from sklearn.preprocessing import StandardScaler

scaler_2017 = StandardScaler()
X_train_2017_scaled = scaler_2017.fit_transform(X_train_2017)

print("Scaling completed.")
print("Scaled training shape:", X_train_2017_scaled.shape)

NameError: name 'X_train_2017' is not defined

In [13]:
import joblib

save_folder = "/content/drive/MyDrive/CIC_IDS_Data/models"
os.makedirs(save_folder, exist_ok=True)

joblib.dump(scaler_2017, save_folder + "/scaler_2017.pkl")
joblib.dump(training_features, save_folder + "/training_features_2017.pkl")

print("Scaler and feature list saved successfully.")

NameError: name 'os' is not defined

In [14]:
# ------------------------------------------------------------
# Train 3 unsupervised models on CICIDS2017 benign traffic
# Models: Isolation Forest, One-Class SVM, Autoencoder
# ------------------------------------------------------------

import os
import gc
import joblib
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


# scale benign training data
# Note: scaler_2017 is refitted here, which might not be desired if it was meant to be the *same* scaler as above.
# If the intent was to use the previously fitted scaler, this line should be `X_train_scaled = scaler_2017.transform(X_train_2017).astype("float32")`
# However, given the context of needing to re-establish the environment, refitting here ensures consistency within this block.
X_train_scaled = scaler_2017.fit_transform(X_train_2017).astype("float32")

print("Training data scaled:", X_train_scaled.shape)


# folder for saving models
model_folder = "/content/drive/MyDrive/CIC_IDS_Data/models"
os.makedirs(model_folder, exist_ok=True)


# ------------------------------------------------------------
# Model 1: Isolation Forest
# ------------------------------------------------------------

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train_scaled)

print("Isolation Forest trained.")


# ------------------------------------------------------------
# Model 2: One-Class SVM
# ------------------------------------------------------------

svm_model = SGDOneClassSVM(
    nu=0.05,
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

svm_model.fit(X_train_scaled)

print("One-Class SVM trained.")


# ------------------------------------------------------------
# Model 3: Autoencoder
# ------------------------------------------------------------

input_dim = X_train_scaled.shape[1]

input_layer = Input(shape=(input_dim,))

x = Dense(64, activation="relu")(input_layer)
x = Dropout(0.2)(x)
x = Dense(32, activation="relu")(x)
x = Dense(16, activation="relu")(x)

x = Dense(32, activation="relu")(x)
x = Dense(64, activation="relu")(x)

output_layer = Dense(input_dim, activation="linear")(x)

autoencoder_model = Model(input_layer, output_layer)

autoencoder_model.compile(
    optimizer="adam",
    loss="mse"
)

early_stop = EarlyStopping(
    monitor="loss",
    patience=3,
    restore_best_weights=True
)

autoencoder_model.fit(
    X_train_scaled,
    X_train_scaled,
    epochs=20,
    batch_size=1024,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

print("Autoencoder trained.")


# ------------------------------------------------------------
# Autoencoder threshold
# ------------------------------------------------------------

train_reconstruction = autoencoder_model.predict(
    X_train_scaled,
    batch_size=1024,
    verbose=1
)

train_error = np.mean(
    np.square(X_train_scaled - train_reconstruction),
    axis=1
)

autoencoder_threshold = np.percentile(train_error, 95)

print("Autoencoder threshold:", autoencoder_threshold)


# ------------------------------------------------------------
# Save models and preprocessing files
# ------------------------------------------------------------

joblib.dump(scaler_2017, model_folder + "/scaler_2017.pkl")
joblib.dump(training_features, model_folder + "/training_features_2017.pkl")

joblib.dump(isolation_model, model_folder + "/isolation_forest_2017.pkl")
joblib.dump(svm_model, model_folder + "/one_class_svm_2017.pkl")

autoencoder_model.save(model_folder + "/autoencoder_2017.keras")
joblib.dump(autoencoder_threshold, model_folder + "/autoencoder_threshold_2017.pkl")

print("All 3 models saved successfully.")

gc.collect()

NameError: name 'X_train_2017' is not defined

In [15]:
# ------------------------------------------------------------
# Test 3 trained models on CICIDS2017
# Same-dataset baseline evaluation
# ------------------------------------------------------------

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd
import numpy as np
import os


# prepare CICIDS2017 test data
X_test_2017 = cicids2017_data.drop(
    columns=["Label", "Label_binary"],
    errors="ignore"
)

X_test_2017 = X_test_2017.select_dtypes(include=["number"])
X_test_2017 = X_test_2017[training_features]

y_test_2017 = cicids2017_data["Label_binary"]

# use the scaler fitted only on CICIDS2017 benign training data
X_test_2017_scaled = scaler_2017.transform(X_test_2017).astype("float32")

print("CICIDS2017 test data ready:", X_test_2017_scaled.shape)


# evaluation function
results = []

def evaluate_model(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    false_positive_rate = fp / (fp + tn)
    false_negative_rate = fn / (fn + tp)
    detection_rate = tp / (tp + fn)

    print("\n------------------------------")
    print(model_name)
    print("------------------------------")
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    results.append({
        "Model": model_name,
        "Dataset": "CICIDS2017 baseline",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "False Positive Rate": false_positive_rate,
        "False Negative Rate": false_negative_rate,
        "Detection Rate": detection_rate
    })


# Model 1: Isolation Forest
isolation_pred = isolation_model.predict(X_test_2017_scaled)
isolation_pred = np.where(isolation_pred == 1, 0, 1)

evaluate_model(
    "Isolation Forest",
    y_test_2017,
    isolation_pred
)


# Model 2: One-Class SVM
svm_pred = svm_model.predict(X_test_2017_scaled)
svm_pred = np.where(svm_pred == 1, 0, 1)

evaluate_model(
    "One-Class SVM",
    y_test_2017,
    svm_pred
)


# Model 3: Autoencoder
ae_reconstruction = autoencoder_model.predict(
    X_test_2017_scaled,
    batch_size=1024,
    verbose=1
)

ae_error = np.mean(
    np.square(X_test_2017_scaled - ae_reconstruction),
    axis=1
)

ae_pred = np.where(
    ae_error > autoencoder_threshold,
    1,
    0
)

evaluate_model(
    "Autoencoder",
    y_test_2017,
    ae_pred
)


# final table
results_2017 = pd.DataFrame(results)

print("\nFinal CICIDS2017 baseline results:")
display(results_2017)


# save results
results_folder = "/content/drive/MyDrive/CICIDS_Data/results"
os.makedirs(results_folder, exist_ok=True)

results_2017.to_csv(
    results_folder + "/cicids2017_baseline_results.csv",
    index=False
)

print("CICIDS2017 baseline results saved.")

NameError: name 'cicids2017_data' is not defined

In [ ]:
data_path = "/content/drive/MyDrive/CICIDS_Data/*.csv"

all_csv_files = glob.glob(data_path, recursive=True)

cicids2017_files = []

for file_path in all_csv_files:
    file_name = os.path.basename(file_path)

    if "pcap_ISCX" in file_name:
        cicids2017_files.append(file_path)

print("CICIDS2017 files found:", len(cicids2017_files))

for file_path in cicids2017_files:
    print(os.path.basename(file_path))

In [1]:
dataframes = []

for file_path in cicids2017_files:
    print("Loading:", os.path.basename(file_path))

    temp_data = pd.read_csv(file_path, low_memory=False)

    # remove extra spaces from column names
    temp_data.columns = temp_data.columns.str.strip()

    dataframes.append(temp_data)

cicids2017_data = pd.concat(dataframes, ignore_index=True)

del dataframes
gc.collect()

print("CICIDS2017 loaded successfully.")
print("Dataset shape:", cicids2017_data.shape)

NameError: name 'cicids2017_files' is not defined

In [2]:
print("Label column exists:", "Label" in cicids2017_data.columns)

print("\nLabel distribution:")
print(cicids2017_data["Label"].value_counts())

NameError: name 'cicids2017_data' is not defined

In [3]:
cicids2017_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values before cleaning:", cicids2017_data.isnull().sum().sum())

cicids2017_data.dropna(inplace=True)

print("Shape after cleaning:", cicids2017_data.shape)
print("Missing values after cleaning:", cicids2017_data.isnull().sum().sum())

NameError: name 'cicids2017_data' is not defined

In [4]:
cicids2017_data["Label_binary"] = cicids2017_data["Label"].apply(
    lambda label: 0 if str(label).upper() == "BENIGN" else 1
)

print("\nBinary label distribution:")
print(cicids2017_data["Label_binary"].value_counts())

NameError: name 'cicids2017_data' is not defined

In [10]:
data_path = "/content/drive/MyDrive/CICIDS_Data/*.csv"

all_csv_files = glob.glob(data_path, recursive=True)

cicids2017_files = []

for file_path in all_csv_files:
    file_name = os.path.basename(file_path)

    if "pcap_ISCX" in file_name:
        cicids2017_files.append(file_path)

print("CICIDS2017 files found:", len(cicids2017_files))

for file_path in cicids2017_files:
    print(os.path.basename(file_path))

CICIDS2017 files found: 8
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [11]:
dataframes = []

for file_path in cicids2017_files:
    print("Loading:", os.path.basename(file_path))

    temp_data = pd.read_csv(file_path, low_memory=False)

    # remove extra spaces from column names
    temp_data.columns = temp_data.columns.str.strip()

    dataframes.append(temp_data)

cicids2017_data = pd.concat(dataframes, ignore_index=True)

del dataframes
gc.collect()

print("CICIDS2017 loaded successfully.")
print("Dataset shape:", cicids2017_data.shape)

Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
CICIDS2017 loaded successfully.
Dataset shape: (2830743, 79)


In [12]:
print("Label column exists:", "Label" in cicids2017_data.columns)

print("\nLabel distribution:")
print(cicids2017_data["Label"].value_counts())

Label column exists: True

Label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [13]:
cicids2017_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values before cleaning:", cicids2017_data.isnull().sum().sum())

cicids2017_data.dropna(inplace=True)

print("Shape after cleaning:", cicids2017_data.shape)
print("Missing values after cleaning:", cicids2017_data.isnull().sum().sum())

Missing values before cleaning: 5734
Shape after cleaning: (2827876, 79)
Missing values after cleaning: 0


In [14]:
cicids2017_data["Label_binary"] = cicids2017_data["Label"].apply(
    lambda label: 0 if str(label).upper() == "BENIGN" else 1
)

print("\nBinary label distribution:")
print(cicids2017_data["Label_binary"].value_counts())


Binary label distribution:
Label_binary
0    2271320
1     556556
Name: count, dtype: int64


In [15]:
data_path = "/content/drive/MyDrive/CICIDS_Data/*.csv"

all_csv_files = glob.glob(data_path, recursive=True)

cicids2017_files = []

for file_path in all_csv_files:
    file_name = os.path.basename(file_path)

    if "pcap_ISCX" in file_name:
        cicids2017_files.append(file_path)

print("CICIDS2017 files found:", len(cicids2017_files))

for file_path in cicids2017_files:
    print(os.path.basename(file_path))

CICIDS2017 files found: 8
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [16]:
dataframes = []

for file_path in cicids2017_files:
    print("Loading:", os.path.basename(file_path))

    temp_data = pd.read_csv(file_path, low_memory=False)

    # remove extra spaces from column names
    temp_data.columns = temp_data.columns.str.strip()

    dataframes.append(temp_data)

cicids2017_data = pd.concat(dataframes, ignore_index=True)

del dataframes
gc.collect()

print("CICIDS2017 loaded successfully.")
print("Dataset shape:", cicids2017_data.shape)

Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
CICIDS2017 loaded successfully.
Dataset shape: (2830743, 79)


In [17]:
print("Label column exists:", "Label" in cicids2017_data.columns)

print("\nLabel distribution:")
print(cicids2017_data["Label"].value_counts())

Label column exists: True

Label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [18]:
cicids2017_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values before cleaning:", cicids2017_data.isnull().sum().sum())

cicids2017_data.dropna(inplace=True)

print("Shape after cleaning:", cicids2017_data.shape)
print("Missing values after cleaning:", cicids2017_data.isnull().sum().sum())

Missing values before cleaning: 5734
Shape after cleaning: (2827876, 79)
Missing values after cleaning: 0


In [19]:

cicids2017_data["Label_binary"] = cicids2017_data["Label"].apply(
    lambda label: 0 if str(label).upper() == "BENIGN" else 1
)

print("\nBinary label distribution:")
print(cicids2017_data["Label_binary"].value_counts())


Binary label distribution:
Label_binary
0    2271320
1     556556
Name: count, dtype: int64


In [20]:
benign_traffic_2017 = cicids2017_data[
    cicids2017_data["Label_binary"] == 0
].copy()

print("Benign traffic shape:", benign_traffic_2017.shape)

Benign traffic shape: (2271320, 80)


In [21]:
X_train_2017 = benign_traffic_2017.drop(
    columns=["Label", "Label_binary"],
    errors="ignore"
)

X_train_2017 = X_train_2017.select_dtypes(include=["number"])

training_features = X_train_2017.columns.tolist()

print("Training data shape:", X_train_2017.shape)
print("Number of training features:", len(training_features))

Training data shape: (2271320, 78)
Number of training features: 78


In [22]:
from sklearn.preprocessing import StandardScaler

scaler_2017 = StandardScaler()
X_train_2017_scaled = scaler_2017.fit_transform(X_train_2017)

print("Scaling completed.")
print("Scaled training shape:", X_train_2017_scaled.shape)

Scaling completed.
Scaled training shape: (2271320, 78)


In [23]:
import joblib

save_folder = "/content/drive/MyDrive/CIC_IDS_Data/models"
os.makedirs(save_folder, exist_ok=True)

joblib.dump(scaler_2017, save_folder + "/scaler_2017.pkl")
joblib.dump(training_features, save_folder + "/training_features_2017.pkl")

print("Scaler and feature list saved successfully.")

Scaler and feature list saved successfully.


In [24]:
X_train_2017
training_features

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

In [25]:
# ------------------------------------------------------------
# Train 3 unsupervised models on CICIDS2017 benign traffic
# Models: Isolation Forest, One-Class SVM, Autoencoder
# ------------------------------------------------------------

import os
import gc
import joblib
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


# scale benign training data
scaler_2017 = StandardScaler()
X_train_scaled = scaler_2017.fit_transform(X_train_2017).astype("float32")

print("Training data scaled:", X_train_scaled.shape)


# folder for saving models
model_folder = "/content/drive/MyDrive/CIC_IDS_Data/models"
os.makedirs(model_folder, exist_ok=True)


# ------------------------------------------------------------
# Model 1: Isolation Forest
# ------------------------------------------------------------

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train_scaled)

print("Isolation Forest trained.")


# ------------------------------------------------------------
# Model 2: One-Class SVM
# ------------------------------------------------------------

svm_model = SGDOneClassSVM(
    nu=0.05,
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

svm_model.fit(X_train_scaled)

print("One-Class SVM trained.")


# ------------------------------------------------------------
# Model 3: Autoencoder
# ------------------------------------------------------------

input_dim = X_train_scaled.shape[1]

input_layer = Input(shape=(input_dim,))

x = Dense(64, activation="relu")(input_layer)
x = Dropout(0.2)(x)
x = Dense(32, activation="relu")(x)
x = Dense(16, activation="relu")(x)

x = Dense(32, activation="relu")(x)
x = Dense(64, activation="relu")(x)

output_layer = Dense(input_dim, activation="linear")(x)

autoencoder_model = Model(input_layer, output_layer)

autoencoder_model.compile(
    optimizer="adam",
    loss="mse"
)

early_stop = EarlyStopping(
    monitor="loss",
    patience=3,
    restore_best_weights=True
)

autoencoder_model.fit(
    X_train_scaled,
    X_train_scaled,
    epochs=20,
    batch_size=1024,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

print("Autoencoder trained.")


# ------------------------------------------------------------
# Autoencoder threshold
# ------------------------------------------------------------

train_reconstruction = autoencoder_model.predict(
    X_train_scaled,
    batch_size=1024,
    verbose=1
)

train_error = np.mean(
    np.square(X_train_scaled - train_reconstruction),
    axis=1
)

autoencoder_threshold = np.percentile(train_error, 95)

print("Autoencoder threshold:", autoencoder_threshold)


# ------------------------------------------------------------
# Save models and preprocessing files
# ------------------------------------------------------------

joblib.dump(scaler_2017, model_folder + "/scaler_2017.pkl")
joblib.dump(training_features, model_folder + "/training_features_2017.pkl")

joblib.dump(isolation_model, model_folder + "/isolation_forest_2017.pkl")
joblib.dump(svm_model, model_folder + "/one_class_svm_2017.pkl")

autoencoder_model.save(model_folder + "/autoencoder_2017.keras")
joblib.dump(autoencoder_threshold, model_folder + "/autoencoder_threshold_2017.pkl")

print("All 3 models saved successfully.")

gc.collect()

Training data scaled: (2271320, 78)
Isolation Forest trained.
One-Class SVM trained.
Epoch 1/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 0.2283
Epoch 2/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1357
Epoch 3/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1157
Epoch 4/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0967
Epoch 5/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1008
Epoch 6/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0799
Epoch 7/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0824
Epoch 8/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0701
Epoch 9/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0785
Epoch 10/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0611
Epoch 11/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0750
Epoch 12/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0561
Epoch 13/20
2219/2219 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0610
Epoch 14/20
2

943

In [ ]:
# ------------------------------------------------------------
# Test 3 trained models on CICIDS2017
# Same-dataset baseline evaluation
# ------------------------------------------------------------

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd
import numpy as np
import os


# prepare CICIDS2017 test data
X_test_2017 = cicids2017_data.drop(
    columns=["Label", "Label_binary"],
    errors="ignore"
)

X_test_2017 = X_test_2017.select_dtypes(include=["number"])
X_test_2017 = X_test_2017[training_features]

y_test_2017 = cicids2017_data["Label_binary"]

# use the scaler fitted only on CICIDS2017 benign training data
X_test_2017_scaled = scaler_2017.transform(X_test_2017).astype("float32")

print("CICIDS2017 test data ready:", X_test_2017_scaled.shape)


# evaluation function
results = []

def evaluate_model(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    false_positive_rate = fp / (fp + tn)
    false_negative_rate = fn / (fn + tp)
    detection_rate = tp / (tp + fn)

    print("\n------------------------------")
    print(model_name)
    print("------------------------------")
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    results.append({
        "Model": model_name,
        "Dataset": "CICIDS2017 baseline",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "False Positive Rate": false_positive_rate,
        "False Negative Rate": false_negative_rate,
        "Detection Rate": detection_rate
    })


# Model 1: Isolation Forest
isolation_pred = isolation_model.predict(X_test_2017_scaled)
isolation_pred = np.where(isolation_pred == 1, 0, 1)

evaluate_model(
    "Isolation Forest",
    y_test_2017,
    isolation_pred
)


# Model 2: One-Class SVM
svm_pred = svm_model.predict(X_test_2017_scaled)
svm_pred = np.where(svm_pred == 1, 0, 1)

evaluate_model(
    "One-Class SVM",
    y_test_2017,
    svm_pred
)


# Model 3: Autoencoder
ae_reconstruction = autoencoder_model.predict(
    X_test_2017_scaled,
    batch_size=1024,
    verbose=1
)

ae_error = np.mean(
    np.square(X_test_2017_scaled - ae_reconstruction),
    axis=1
)

ae_pred = np.where(
    ae_error > autoencoder_threshold,
    1,
    0
)

evaluate_model(
    "Autoencoder",
    y_test_2017,
    ae_pred
)


# final table
results_2017 = pd.DataFrame(results)

print("\nFinal CICIDS2017 baseline results:")
display(results_2017)


# save results
results_folder = "/content/drive/MyDrive/CICIDS_Data/results"
os.makedirs(results_folder, exist_ok=True)

results_2017.to_csv(
    results_folder + "/cicids2017_baseline_results.csv",
    index=False
)

print("CICIDS2017 baseline results saved.")